# Run Experiment & Cache Results

Runs the inverse optimization with pre-tuned hyperparameters and saves all outputs
to `results/<experiment_id>/` so the figure notebooks can load them without re-running.

Re-run this notebook only when you want to change the hyperparameters or process more images.
Running it again with the same config is a no-op (the cache directory already exists).

In [ ]:
%%capture
import os, subprocess, sys

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")

    REPO = "/content/drive/MyDrive/"

    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "-r", f"{REPO}/notebooks/requirements.txt",
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "--no-deps", "-e", REPO,
    ])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "dict-hash"])
else:
    subprocess.check_call(["uv", "pip", "install", "-q", "python-dotenv", "dict-hash"])

In [2]:
!pwd

/Volumes/T7/projs/thesis/figures


## 0. Environment setup

**Colab:** mounts Drive, installs dependencies.  
**Local:** loads `.env` from repo root.

In [ ]:
import os

IN_COLAB = "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    DRIVE_ROOT = "/content/drive/MyDrive/thesis"  # <-- edit this

    os.environ["IMAGE_PATH"] = f"{DRIVE_ROOT}/chexpert_plus/PNG/PNG"
    os.environ["META_PATH"]  = f"{DRIVE_ROOT}/chexpert_plus/df_chexpert_plus_240401.csv"
    os.environ["MASK_DIR"]   = f"{DRIVE_ROOT}/chexpert_plus/masks/"
    os.environ["CACHE_DIR"]  = f"{DRIVE_ROOT}/torchxrayvision/"
    os.environ["OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs/"

    REPO = DRIVE_ROOT
else:
    from dotenv import load_dotenv
    load_dotenv("../.env")
    REPO = os.path.abspath("..")

GT_DIR = os.path.join(REPO, "data", "Processed vs unprocessed real GE scanner")

print("IMAGE_PATH:", os.environ.get("IMAGE_PATH", "<not set>"))
print("CACHE_DIR: ", os.environ.get("CACHE_DIR",  "<not set>"))
print("GT_DIR:    ", GT_DIR)

In [ ]:
import sys

EXPERIMENTS_DIR = os.path.join(REPO, "experiments")
if EXPERIMENTS_DIR not in sys.path:
    sys.path.insert(0, EXPERIMENTS_DIR)

print("Experiments dir:", EXPERIMENTS_DIR)


## 1. Hyperparameters

Pre-tuned configuration. `experiment_id` is a content hash so the same config always maps to the same cache directory.

In [ ]:
from models import ExperimentArgs, default_params
from dict_hash import sha256

conf = {
    "PRNGKey": 0,
    "constant_weights": False,
    "data_fidelity": "l2",
    "eps": 1e-07,
    "gmse_weight": 0.0,
    "high_min_enhancement": 0.11737927252785224,
    "high_sigma": 10,
    "log_samples": 10,
    "low_min_enhancement": 0.16897921208762814,
    "lr": 0.029666762984711156,
    "max_enhancement": 0.8,
    "max_sigma": 10,
    "momentum": 0.9,
    "n_steps": 300,
    "optimizer": "amsgrad",
    "prior_weight": 1.8310279207471134e-05,
    "segmentation_th": 0.6,
    "sigma": 1,
    "smooth_metric": "none",
    "tm_init_params": ("naive", None),
    "total_variation": 5.271572986163628e-07,
    "use_band_similarity": False,
    "variant": "double-band",
    "weight_decay": 0.0001,
    "windowing_type": "sigmoid",
}

BATCH_SIZE  = 4
NUM_BATCHES = 64

config = ExperimentArgs(**conf)
w0 = default_params(config)

experiment_id = sha256({**conf, "sample_size": NUM_BATCHES * BATCH_SIZE})
RESULTS_DIR = os.path.join(REPO, "figures", "results")
cache_dir = os.path.join(RESULTS_DIR, experiment_id[:7])
os.makedirs(cache_dir, exist_ok=True)

print(f"Cache dir: {cache_dir}")
print(f"Sample size: {NUM_BATCHES * BATCH_SIZE} images")

## 2. Load dataset & compute priors

In [ ]:
from segmentation_utils import get_priors

seg_cache_dir = os.environ["CACHE_DIR"]
gt_tm_path   = os.path.join(GT_DIR, "Z01-oprocess.tif")
gt_proc_path = os.path.join(GT_DIR, "Z01-process.tif")

# value_ranges: used as optimization constraints (excludes collimated border region)
_, value_ranges = get_priors(
    seg_cache_dir,
    collimated_region_bound=0.4,
    real_tm_paths=[gt_tm_path],
    fwd_img_paths=[gt_proc_path],
)

# priors: full-image ranges used for figure annotations
_, priors = get_priors(
    seg_cache_dir,
    real_tm_paths=[gt_tm_path],
    fwd_img_paths=[gt_proc_path],
)

print("value_ranges:", value_ranges)
print("priors:      ", priors)

In [ ]:
from chest_xray_sim.data.dataset import Loader
from chest_xray_sim.data.segmentation_dataset import SegmentationChexpertDataset

dataset = SegmentationChexpertDataset(
    root=os.environ["IMAGE_PATH"],
    meta_dir=os.environ["META_PATH"],
    mask_dir=os.environ["MASK_DIR"],
    cache_dir=seg_cache_dir,
    split="train",
    frontal_lateral="Frontal",
)

loader = Loader(
    dataset,
    batch_size=BATCH_SIZE,
    collate_fn=SegmentationChexpertDataset.collate,
    shuffle=False,
)

print(f"Dataset size: {len(dataset)}")
print(f"Processing {NUM_BATCHES} batches × {BATCH_SIZE} images = {NUM_BATCHES * BATCH_SIZE} total")

## 3. Run optimization & save

Skips running if all output files already exist in `cache_dir`.

In [ ]:
import jax.numpy as jnp
import joblib
from chest_xray_sim.inverse.eval import EvalMetrics
from chest_xray_sim.segmentation.model import MASK_GROUPS
from replicate import execute

_required = ["imgs", "masks", "txm", "pred", "weights", "metrics", "priors", "value_ranges", "seg_labels"]
_all_exist = all(os.path.exists(os.path.join(cache_dir, f"{f}.joblib")) for f in _required)

if _all_exist:
    print(f"All outputs already cached in {cache_dir} — skipping optimization.")
else:
    metrics_list, imgs_list, masks_list, txm_list, weights_list, pred_list = execute(
        loader, config, w0, value_ranges, save_dir=None, batches=NUM_BATCHES,
    )

    imgs    = jnp.concatenate(imgs_list, axis=0)
    masks   = jnp.concatenate(masks_list, axis=0)
    txm     = jnp.concatenate(txm_list, axis=0)
    pred    = jnp.concatenate(pred_list, axis=0)
    weights = {k: jnp.concatenate([w[k] for w in weights_list], axis=0) for k in weights_list[0].keys()}
    metrics = EvalMetrics.merge(metrics_list)

    joblib.dump(imgs,         os.path.join(cache_dir, "imgs.joblib"))
    joblib.dump(masks,        os.path.join(cache_dir, "masks.joblib"))
    joblib.dump(txm,          os.path.join(cache_dir, "txm.joblib"))
    joblib.dump(pred,         os.path.join(cache_dir, "pred.joblib"))
    joblib.dump(weights,      os.path.join(cache_dir, "weights.joblib"))
    joblib.dump(metrics,      os.path.join(cache_dir, "metrics.joblib"))
    joblib.dump(priors,       os.path.join(cache_dir, "priors.joblib"))
    joblib.dump(value_ranges, os.path.join(cache_dir, "value_ranges.joblib"))
    joblib.dump(list(MASK_GROUPS), os.path.join(cache_dir, "seg_labels.joblib"))

    print(f"Saved {imgs.shape[0]} images to {cache_dir}")

In [ ]:
# Quick sanity check — load and print shapes
imgs    = joblib.load(os.path.join(cache_dir, "imgs.joblib"))
txm     = joblib.load(os.path.join(cache_dir, "txm.joblib"))
pred    = joblib.load(os.path.join(cache_dir, "pred.joblib"))
masks   = joblib.load(os.path.join(cache_dir, "masks.joblib"))
priors  = joblib.load(os.path.join(cache_dir, "priors.joblib"))
seg_labels = joblib.load(os.path.join(cache_dir, "seg_labels.joblib"))

print(f"imgs:   {imgs.shape}")
print(f"txm:    {txm.shape}")
print(f"pred:   {pred.shape}")
print(f"masks:  {masks.shape}  (regions: {seg_labels})")
print(f"priors: {priors}")
print(f"\nFigure notebooks will load from: {cache_dir}")